# BSV Batch Processor — Text/Image GT

Runs image extraction (loosened thresholds: MIN_IMAGE_AREA, last-page
recovery, footer filtering) across the 2,404-PDF reserved GT pool.
Table extraction is skipped entirely, this run is text/image and
image/caption focused only.

Output:
- `bsv_dataset_gt_2404.jsonl`: one JSON record per bulletin section, one line per record
- `bsv_images_gt_2404/` — all extracted image files
- `bsv_pairs_gt_textimage_900.jsonl`: sampled text_image + image_caption pairs for manual validation

**Prerequisites:** run `bsv_parser_subsections_v3.ipynb` first (or ensure
`bsv_parser.py` is in the same directory).



## 1. Install dependencies

In [1]:
import sys
!{sys.executable} -m pip install pymupdf pdfplumber --break-system-packages -q
print("Done")


Done


Shuffling the ~20,000 PDF URLs

In [2]:
"""
import random

url_file = "/projects/agrifutur/yabusaqe/BSV Sample Construction/PDFs Download/PDF_URLS.txt""

with open(url_file) as f:
    urls = f.readlines()

random.seed(42)
random.shuffle(urls)

with open(url_file, "w") as f:
    f.writelines(urls)

print(f"Shuffled {len(urls)} URLs")
"""

'\nimport random\n\nurl_file = "/projects/agrifutur/yabusaqe/BSV Sample Construction/PDF_URLS.txt"\n\nwith open(url_file) as f:\n    urls = f.readlines()\n\nrandom.seed(42)\nrandom.shuffle(urls)\n\nwith open(url_file, "w") as f:\n    f.writelines(urls)\n\nprint(f"Shuffled {len(urls)} URLs")\n'

## 2. Imports

In [3]:
import os
print(os.getcwd())
print(os.listdir(os.getcwd()))

/projects/agrifutur/yabusaqe/BSV Sample Construction
['test_bulletin.pdf', '.ipynb_checkpoints', 'bsv_parser.ipynb', '__pycache__', 'PDF_URLS.txt', 'Previous Iterations Data', 'bsv_heuristics_analysis.ipynb', 'bsv_download.ipynb', 'bsv_parser_subsections.ipynb', 'bsv_parser_subsections_v2.ipynb', 'french_words.txt', 'recall_approx.py', 'reserve_gt_pdfs.py', 'gt_reserved_pdfs.jsonl', 'Old Viewers', 'bsv_batch.ipynb', 'bsv_pair_constructor.py', 'bsv_parser.py', 'bsv_parser_subsections_v3.ipynb', 'bsv_reviewer.html', 'generate_reviewer.py']


In [4]:
import json, os

nb = json.load(open("/projects/agrifutur/yabusaqe/BSV Sample Construction/Parsers/bsv_parser_subsections_v3.ipynb"))
code_cells = [c for c in nb["cells"] if c["cell_type"] == "code"]

with open("/projects/agrifutur/yabusaqe/BSV Sample Construction/Parsers/bsv_parser.ipynb", "w") as f:
    for cell in code_cells:
        src = "".join(cell["source"])
        # skip install and test cells
        if any(skip in src for skip in [
            "pip install",
            "parse_pdf(TEST_PDF",
            "urllib.request.urlretrieve",
            "fitz.open(TEST_PDF",
            "fitz.open(TEST_CHARENTES",
            "parse_structure(TEST_PDF",
            "os.listdir(PDF_DIR)",
            "parse_pdf(TEST_PDF2",
            "parse_pdf(TEST_CHARENTES",
            "print(SIZE_THRESHOLD)",
]):
            continue
        f.write(src + "\n\n")

print("Created bsv_parser.py")
print(os.path.exists("bsv_parser.py"))

Created bsv_parser.py
True


In [5]:
import sys
if "bsv_parser" in sys.modules:
    import importlib
    import bsv_parser
    importlib.reload(bsv_parser)
print("bsv_parser reloaded" if "bsv_parser" in sys.modules else "bsv_parser not yet imported")

bsv_parser not yet imported


In [6]:
"""
import json, os

nb = json.load(open("bsv_parser.ipynb"))
code_cells = [c for c in nb["cells"] if c["cell_type"] == "code"]

with open("bsv_parser.py", "w") as f:
    for cell in code_cells:
        src = "".join(cell["source"])
        if any(skip in src for skip in [
            "pip install",
            "TEST_PDF",
            "Bold size distribution",
        ]):
            continue
        f.write(src + "\n\n")

print("Done")
fns = __import__("re").findall(r'^def (\w+)', open("bsv_parser.py").read(), __import__("re").MULTILINE)
print(fns)
"""

<>:1: SyntaxWarning: invalid escape sequence '\w'
<>:1: SyntaxWarning: invalid escape sequence '\w'
/tmp/ipykernel_2178509/2007335901.py:1: SyntaxWarning: invalid escape sequence '\w'
  """


'\nimport json, os\n\nnb = json.load(open("bsv_parser.ipynb"))\ncode_cells = [c for c in nb["cells"] if c["cell_type"] == "code"]\n\nwith open("bsv_parser.py", "w") as f:\n    for cell in code_cells:\n        src = "".join(cell["source"])\n        if any(skip in src for skip in [\n            "pip install",\n            "TEST_PDF",\n            "Bold size distribution",\n        ]):\n            continue\n        f.write(src + "\n\n")\n\nprint("Done")\nfns = __import__("re").findall(r\'^def (\\w+)\', open("bsv_parser.py").read(), __import__("re").MULTILINE)\nprint(fns)\n'

In [7]:
import re
with open("bsv_parser.py") as f:
    src = f.read()
fns = re.findall(r'^def (\w+)', src, re.MULTILINE)
print(fns)

['is_footer', 'is_bad_header', 'is_credit_line', 'is_chart_noise', 'extract_metadata', 'parse_structure', 'extract_text', 'split_into_sections', 'extract_images', 'match_captions', 'count_raw_images', 'count_raw_tables', 'repair_table_via_docx', 'extract_tables', 'assign_to_sections', 'parse_pdf']


In [8]:
import os, re, sys, json, time, threading, tempfile, urllib.request
import pdfplumber
from pathlib import Path
from concurrent.futures import ThreadPoolExecutor, as_completed
sys.path.insert(0, os.getcwd())
import bsv_parser
from bsv_parser import parse_pdf
bsv_parser.MIN_IMAGE_AREA = 30000   # was 200,000, loosened for GT only

def loose_extract_tables(pdf_path):
    result = []
    with pdfplumber.open(pdf_path) as pdf:
        for page_num, page in enumerate(pdf.pages, 1):
            if page_num in bsv_parser.SKIP_PAGES:
                continue
            for tbl_obj in page.find_tables():
                table = tbl_obj.extract()
                if not table or len(table) < 3:
                    continue
                n_rows = len(table)
                n_cols = max(len(row) for row in table)
                if n_cols < 2:
                    continue
                flat = [c for row in table for c in row if c]
                avg_cell_len = sum(len(c) for c in flat) / max(len(flat), 1)
                if avg_cell_len > 200:
                    continue
                all_cells = [c for row in table for c in row]
                empty_pct = sum(1 for c in all_cells if not c or str(c).strip() == '') / max(len(all_cells), 1)
                if empty_pct > 0.92:            # was 0.5, loosened for GT only
                    continue
                non_empty_count = sum(1 for c in all_cells if c and str(c).strip())
                if non_empty_count < 5:
                    continue
                single_col_row_pct = None
                if len(table) > 1:
                    single_column_rows = sum(1 for row in table if sum(1 for c in row if c and str(c).strip()) == 1)
                    single_col_row_pct = single_column_rows / len(table)
                    if single_col_row_pct > 0.60:
                        continue
                
                consistent_cols = None
                if n_rows > 2 and n_cols > 1:
                    col_fill = {}
                    for row in table:
                        for ci, cell in enumerate(row):
                            if cell and str(cell).strip():
                                col_fill[ci] = col_fill.get(ci, 0) + 1
                    if col_fill:
                        max_fill = max(col_fill.values())
                        consistent_cols = sum(1 for v in col_fill.values() if v >= max_fill * 0.5)
                        if consistent_cols <= 1:
                            continue
                repaired = bsv_parser.repair_table_via_docx(pdf_path, page_num)
                if repaired:
                    table = repaired

                v_rules = sorted(set(
                    round(edge["x0"]) for edge in page.edges
                    if edge["orientation"] == "v"
                    and edge["top"] >= tbl_obj.bbox[1] - 2
                    and edge["bottom"] <= tbl_obj.bbox[3] + 2
                ))

                result.append({"page": page_num, "rows": table, "bbox": tbl_obj.bbox,
                                "n_rows": n_rows, "n_cols": n_cols, "avg_cell_len": avg_cell_len,
                                "empty_pct": empty_pct, "single_col_row_pct": single_col_row_pct,
                                "consistent_cols": consistent_cols, "vertical_rules": v_rules})
    return result

_strict_parse_structure = bsv_parser.parse_structure   # save the real one before overwriting

def loose_parse_structure(pdf_path, table_bboxes):
    s_raw, i_raw, text_els = _strict_parse_structure(pdf_path, table_bboxes)

    import fitz
    doc = fitz.open(pdf_path)
    total_pages = doc.page_count

    if total_pages >= 2:
        last_page_num = total_pages
        page = doc[last_page_num - 1]
        page_height = page.rect.height

        already_seen_xrefs = {img["xref"] for img in i_raw}

        img_info_list = {item["xref"]: item for item in page.get_image_info(hashes=False, xrefs=True)}
        seen_here = set()
        for img in page.get_images(full=True):
            xref = img[0]
            if xref in already_seen_xrefs or xref in seen_here:
                continue
            seen_here.add(xref)
            info = img_info_list.get(xref)
            if not info or not info.get("bbox"):
                continue
            bbox = info["bbox"]
            top, bottom = bbox[1], bbox[3]

            if top >= 0.90 * page_height:
                continue

            i_raw.append({
                "page": last_page_num, "top": top, "bottom": bottom,
                "left": bbox[0], "right": bbox[2],
                "xref": xref, "page_height": page_height,
            })

    doc.close()

    # Drop any image sitting in the bottom ~10% of its OWN page, on every
    # page, not just the last one. Catches recurring footer banners that
    # don't share one xref across pages, so the strict parser's xref-based
    # recurring-logo check never saw them in the first place.
    i_raw = [
        img for img in i_raw
        if not (img.get("page_height") and img["top"] >= 0.90 * img["page_height"])
    ]

    return s_raw, i_raw, text_els


bsv_parser.parse_structure = loose_parse_structure

bsv_parser.extract_tables = loose_extract_tables
# TODAY ONLY: skip table extraction entirely, we only want text/image pairs,
# and this is the slowest part of parsing (pdfplumber table search + merge
# repair conversions). Tomorrow, when tables are needed again, just remove
# this line, loose_extract_tables above stays untouched either way.
def skip_tables(pdf_path):
    return []

bsv_parser.extract_tables = skip_tables

import os
os.environ["SKIP_OCR"] = "1"

/projects/agrifutur/yabusaqe/BSV Sample Construction
['test_bulletin.pdf', '.ipynb_checkpoints', 'bsv_parser.ipynb', '__pycache__', 'PDF_URLS.txt', 'Previous Iterations Data', 'bsv_heuristics_analysis.ipynb', 'bsv_download.ipynb', 'bsv_parser_subsections.ipynb', 'bsv_parser_subsections_v2.ipynb', 'french_words.txt', 'recall_approx.py', 'reserve_gt_pdfs.py', 'gt_reserved_pdfs.jsonl', 'Old Viewers', 'bsv_batch.ipynb', 'bsv_pair_constructor.py', 'bsv_parser.py', 'bsv_parser_subsections_v3.ipynb', 'bsv_reviewer.html', 'generate_reviewer.py']
p1 size=7.0 last='e' text='Animateur filière'
p1 size=7.0 last='e' text='Chambre d’agriculture de'
p1 size=7.0 last='e' text='la Charente-Maritime'
p1 size=7.0 last='n' text='Directeur de publication'
p1 size=7.0 last='n' text='Supervision'
p1 size=6.0 last='e' text='Reproduction intégrale'
p1 size=6.0 last='.' text='de ce bulletin autorisée.'
p1 size=6.0 last='e' text='Reproduction partielle'
p1 size=6.0 last='n' text='autorisée avec la mention'
p1 si

## 3. Configuration

In [9]:
MANIFEST     = "/projects/agrifutur/yabusaqe/BSV Sample Construction/Ground Truth/gt_reserved_pdfs.jsonl"
OUT_JSONL    = "/projects/agrifutur/yabusaqe/BSV Sample Construction/Ground Truth/bsv_dataset_gt_2404.jsonl"
OUT_IMGS     = "/projects/agrifutur/yabusaqe/BSV Sample Construction/Ground Truth/bsv_images_gt_2404"
WORKERS      = 20
LIMIT        = None
CHECKPOINT   = "/projects/agrifutur/yabusaqe/BSV Sample Construction/Ground Truth/processed_urls_gt_2404.txt""
MIN_PDF_SIZE = 5000

## 4. Load URLs from SPARQL results file

In [10]:
"""
import random
import time


all_pdf_files = sorted([
    f for f in os.listdir(PDF_DIR)
    if f.endswith('.pdf')
])

random.seed(time.time())
pdf_files = random.sample(all_pdf_files, min(200, len(all_pdf_files)))

print(f"Total PDFs in folder : {len(all_pdf_files)}")
print(f"Randomly sampled     : {len(pdf_files)}")
print(f"First                : {pdf_files[0]}")
print(f"Last                 : {pdf_files[-1]}")
"""

import json as _json

pdf_paths = {}
with open(MANIFEST, encoding="utf-8") as f:
    for line in f:
        entry = _json.loads(line)
        pdf_paths[entry["filename"]] = os.path.join(entry["source_dir"], entry["filename"])

pdf_files = sorted(pdf_paths.keys())
print(f"Loaded {len(pdf_files)} reserved GT PDFs from manifest")
print(f"First: {pdf_files[0]}")
print(f"Last:  {pdf_files[-1]}")

Loaded 2404 reserved GT PDFs from manifest
First: 01_BSV_ARBO_180316_cle8a7466.pdf
Last:  spip.phpactionacceder_documentarg878clebf74b13b51cb45eee0cd755d3dd4462bdf7a6b69filepdf2Fcc_AAP_PNA-VF.pdf


## 5. Download helper

Defines `download_pdf` (retries up to `MAX_RETRIES`, checks minimum file size) and `process_one` (downloads to a temp directory, calls `parse_pdf`, returns records or empty list on failure).


In [11]:
def download_pdf(url, dest):
    for attempt in range(MAX_RETRIES + 1):
        try:
            req = urllib.request.Request(
                url, headers={"User-Agent": "BSV-Dataset-Builder/1.0"})
            with urllib.request.urlopen(req, timeout=DOWNLOAD_TIMEOUT) as resp:
                data = resp.read()
            if len(data) < MIN_PDF_SIZE:
                return False
            with open(dest, "wb") as f:
                f.write(data)
            return True
        except Exception:
            if attempt < MAX_RETRIES:
                time.sleep(2 ** attempt)
    return False


def process_one(filename):
    pdf_path = pdf_paths[filename]
    if os.path.getsize(pdf_path) < MIN_PDF_SIZE:
        return filename, []
    try:
        records = parse_pdf(pdf_path, OUT_IMGS, url=None)
        return filename, records
    except Exception as e:
        return filename, []

## 6. Run batch

 Batch run: loads the checkpoint to skip already-processed URLs. Opens `OUT_JSONL` in append mode. Runs `process_one` across remaining URLs with `ThreadPoolExecutor`. After each success, writes section records to the JSONL and appends the URL to the checkpoint under a separate lock. Prints progress every 100 PDFs and at the first 5.


In [12]:
os.makedirs(OUT_IMGS, exist_ok=True)
write_lock      = threading.Lock()
checkpoint_lock = threading.Lock()
counts = {"processed": 0, "sections": 0, "errors": 0}

if os.path.exists(CHECKPOINT):
    with open(CHECKPOINT) as f:
        done_files = set(f.read().splitlines())
    print(f"Resuming: {len(done_files)} PDFs already processed")
else:
    done_files = set()

files_to_run = [f for f in pdf_files if f not in done_files]
print(f"Remaining: {len(files_to_run)} PDFs to process")

with open(OUT_JSONL, "a", encoding="utf-8") as out_f:
    with ThreadPoolExecutor(max_workers=WORKERS) as executor:
        futures = {executor.submit(process_one, fname): fname for fname in files_to_run}
        for future in as_completed(futures):
            fname, records = future.result()
            with write_lock:
                for rec in records:
                    out_f.write(json.dumps(rec, ensure_ascii=False) + "\n")
                counts["sections"] += len(records)
                counts["processed"] += 1
                if not records:
                    counts["errors"] += 1
            with checkpoint_lock:
                with open(CHECKPOINT, "a") as ck:
                    ck.write(fname + "\n")
            if counts["processed"] % 100 == 0 or counts["processed"] <= 5:
                print(f"[{counts['processed']}/{len(files_to_run)}] "
                      f"sections={counts['sections']} errors={counts['errors']}")

print(f"Done. Processed={counts['processed']} Sections={counts['sections']} Errors={counts['errors']}")

Remaining: 2404 PDFs to process
[1/2404] sections=1 errors=0
[2/2404] sections=2 errors=0
[3/2404] sections=3 errors=0
[4/2404] sections=4 errors=0
[5/2404] sections=5 errors=0
[100/2404] sections=1152 errors=0
[200/2404] sections=2348 errors=0


Cannot set non-stroke color: 2 components specified, but only 1 (grayscale), 3 (RGB), and 4 (CMYK) are supported


[300/2404] sections=3152 errors=0
[400/2404] sections=4042 errors=0
[500/2404] sections=5144 errors=0
[600/2404] sections=5724 errors=0
[700/2404] sections=7322 errors=0
[800/2404] sections=8310 errors=0
[900/2404] sections=9439 errors=0
[1000/2404] sections=10845 errors=0
[1100/2404] sections=12680 errors=0
[1200/2404] sections=13671 errors=0
[1300/2404] sections=15090 errors=0
[1400/2404] sections=16303 errors=0
[1500/2404] sections=17764 errors=0
[1600/2404] sections=18198 errors=1
[1700/2404] sections=19279 errors=1
[1800/2404] sections=19943 errors=1
[1900/2404] sections=20539 errors=1
[2000/2404] sections=21520 errors=1


Cannot set CMYK stroke color because not all values in (6, /b'_\x9c\xb1D\xf9\xed\xf2\x1aB\xea\xfb\xaemk\x82\x9dv\x06\x1cN\'k\xdd\xe3\xf7\xefD\xde5v\x9c^"D\xe3d\x81\x10\xfc\xd5Up\xe8j', /b'\x84\x90R\xe6\x92', /b'\xe3?J\x8b\xae\xdd\xdd&\xba\x8d\xd7\xf3\xe6\xe7\xaaV\xf3|\xb5\xce\x1e\x87\x11==\xff\x15\xc8\xf3b\xf6\x8d') can be parsed as floats


[2100/2404] sections=22711 errors=1
[2200/2404] sections=23799 errors=1
[2300/2404] sections=24369 errors=1
[2400/2404] sections=25833 errors=1
Done. Processed=2404 Sections=25913 Errors=1


## 7. Inspect output

 Reads and prints the first 5 records from the output JSONL as a quick sanity check.


In [13]:
with open(OUT_JSONL, encoding="utf-8") as f:
    sample = [json.loads(l) for l in f.readlines()[:5]]

for rec in sample:
    print(f"bulletin_id   : {rec['bulletin_id']}")
    print(f"crop          : {rec['crop']}")
    print(f"date          : {rec['date']}")
    print(f"section_title : {rec['section_title']}")
    print(f"text preview  : {rec['text'][:120]!r}")
    print(f"images        : {len(rec['images'])}")
    print(f"tables        : {len(rec['tables'])}")
    print()


bulletin_id   : 1.bsv-35_septembre2022-resume
crop          : None
date          : None
section_title : Document
text preview  : 'Direction de publication\n:\nPhilippe Gout, directeur de la DAAF Mayotte & Christophe Bretagne, directeur de l’EPN\nCoconi\n'
images        : 0
tables        : 0

bulletin_id   : 01_BSV_ARBO_180316_cle8a7466
crop          : Arboriculture
date          : None
section_title : Document
text preview  : 'n\nAnimateurs :\n,\nChambre d’agriculture Alsace\nParticipants\n:\nArboriculture\nPOIRIER\nPsylle :\npontes\nfaibles\nPOMMIER\nAntho'
images        : 0
tables        : 0

bulletin_id   : 02_BSV_ARBO_230316_cle8d1563
crop          : Arboriculture
date          : None
section_title : Document
text preview  : 'n\nAnimateurs :\n,\nChambre d’agriculture Alsace\nParticipants\n:\nArboriculture\nPOIRIER\nPsylle :\nactivité\nfaible\nPOMMIER\nAnth'
images        : 0
tables        : 0

bulletin_id   : 1204_rodez_agglo_cle4e1771
crop          : None
date          : None
sect

Counts total lines in the JSONL to report total sections extracted:

In [14]:
import re
with open("bsv_parser.py") as f:
    src = f.read()
fns = re.findall(r'^def (\w+)', src, re.MULTILINE)
print(fns)

['is_footer', 'is_bad_header', 'is_credit_line', 'is_chart_noise', 'extract_metadata', 'parse_structure', 'extract_text', 'split_into_sections', 'extract_images', 'match_captions', 'count_raw_images', 'count_raw_tables', 'repair_table_via_docx', 'extract_tables', 'assign_to_sections', 'parse_pdf']


In [15]:
with open(OUT_JSONL) as f:
    total = sum(1 for _ in f)
print(f"Total sections: {total}")

Total sections: 25913


 Draws a reproducible random 500-section sample  from the full JSONL and saves it to `bsv_sample_500.jsonl` for manual curation.section.

In [16]:
import json

with open("/projects/agrifutur/yabusaqe/BSV Sample Construction/Ground Truth/bsv_dataset_gt_2404.jsonl", encoding="utf-8") as f:
    seen = set()
    total_found_img = total_kept_img = 0
    total_found_tbl = total_kept_tbl = 0
    for line in f:
        rec = json.loads(line)
        if rec["bulletin_id"] in seen:
            continue  # only count once per bulletin, meta is repeated per section
        seen.add(rec["bulletin_id"])
        total_found_img += rec.get("images_found_total", 0)
        total_kept_img  += rec.get("images_kept", 0)
        total_found_tbl += rec.get("tables_found_total", 0)
        total_kept_tbl  += rec.get("tables_kept", 0)

print(f"Image recall: {total_kept_img}/{total_found_img} "
      f"({100*total_kept_img/max(total_found_img,1):.1f}%)")
print(f"Table recall: {total_kept_tbl}/{total_found_tbl} "
      f"({100*total_kept_tbl/max(total_found_tbl,1):.1f}%)")

Image recall: 18485/91976 (20.1%)
Table recall: 0/19477 (0.0%)


In [17]:
import json, random
from bsv_pair_constructor import generate_pairs

all_pairs = []
with open("/projects/agrifutur/yabusaqe/BSV Sample Construction/Ground Truth/bsv_dataset_gt_2404.jsonl", encoding="utf-8") as f:
    for line in f:
        rec = json.loads(line)
        pairs = generate_pairs(rec)
        all_pairs.extend(p for p in pairs if p["pair_type"] in ("text_image", "image_caption"))

print(f"Total text_image + image_caption pairs across all 2,404 PDFs: {len(all_pairs)}")

random.seed(42)
sample_900 = random.sample(all_pairs, min(900, len(all_pairs)))

with open("/projects/agrifutur/yabusaqe/BSV Sample Construction/Ground Truth/bsv_pairs_gt_textimage_900.jsonl", "w", encoding="utf-8") as f:
    for p in sample_900:
        f.write(json.dumps(p, ensure_ascii=False) + "\n")
print(f"Sampled {len(sample_900)} pairs -> bsv_pairs_gt_textimage_900.jsonl")

Total text_image + image_caption pairs across all 2,404 PDFs: 5907
Sampled 900 pairs -> bsv_pairs_gt_textimage_900.jsonl


In [19]:
import re
with open("/projects/agrifutur/yabusaqe/BSV Sample Construction/Scripts/generate_reviewer.py", encoding="utf-8") as f:
    src = f.read()
src = re.sub(r'PAIRS_FILE = ".*"', 'PAIRS_FILE = "/projects/agrifutur/yabusaqe/BSV Sample Construction/Ground Truth/bsv_pairs_gt_textimage_900.jsonl"', src)
src = re.sub(r'IMG_DIR\s*= ".*"', 'IMG_DIR    = "bsv_images_gt_2404"', src)
src = re.sub(r'HTML_OUT\s*= ".*"', 'HTML_OUT   = "/projects/agrifutur/yabusaqe/BSV Sample Construction/Ground Truth/bsv_images_gt_2404"', src)
with open("generate_reviewer_gt_textimage.py", "w", encoding="utf-8") as f:
    f.write(src)